In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, warnings
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
!pip install -q lime
from lime import lime_image
from skimage.segmentation import mark_boundaries
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_ROOT = '/content/drive/MyDrive/MRI_Brain_Tumor_Project'
CLAHE_DIR    = f'{PROJECT_ROOT}/data/clahe_processed'
IMG_SIZE     = (224, 224)
CLASS_NAMES  = ['glioma', 'meningioma', 'notumor', 'pituitary']
PRED_DIR     = f'{PROJECT_ROOT}/results/predictions'
FIG_DIR      = f'{PROJECT_ROOT}/results/figures'

print("Loading MobileNetV2 model...")
mobilenet_model = tf.keras.models.load_model(
    f'{PROJECT_ROOT}/models/checkpoints/mobilenetv2_clean.keras'
)
preprocess_fn = tf.keras.applications.mobilenet_v2.preprocess_input

y_true      = np.load(f'{PRED_DIR}/y_true.npy')
uncertainty = np.load(f'{PRED_DIR}/uncertainty_mobilenet.npy')

# Same representative sample indices as the EfficientNetB3 notebook, so the
# two XAI panels are directly comparable image-for-image.
sample_indices = {'glioma':32,'meningioma':570,'notumor':1195,'pituitary':1202}

def load_img(cls_name, img_idx):
    class_order  = sorted(os.listdir(f'{CLAHE_DIR}/Testing'))
    cls_position = class_order.index(cls_name)
    local_idx    = img_idx - cls_position * 400
    files        = sorted(os.listdir(f'{CLAHE_DIR}/Testing/{cls_name}'))
    img = cv2.imread(f'{CLAHE_DIR}/Testing/{cls_name}/{files[local_idx]}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, IMG_SIZE)

data = {}
for cls_name, idx in sample_indices.items():
    img   = load_img(cls_name, idx)
    img_p = preprocess_fn(img.astype(np.float32))[np.newaxis]
    probs = mobilenet_model(img_p, training=False).numpy()[0]
    pc    = int(np.argmax(probs))
    data[cls_name] = {'img': img, 'img_p': img_p, 'pred': pc,
                       'conf': float(probs[pc]), 'unc': float(uncertainty[idx])}
    print(f"  {cls_name}: pred={CLASS_NAMES[pc]}, conf={probs[pc]:.3f}, "
          f"unc={uncertainty[idx]:.4f}"
          + ("  <-- MISCLASSIFIED, note this before comparing to EfficientNetB3" if pc != idx else ""))

print("\nAll 4 images loaded. Starting XAI...\n")


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 20.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Loading MobileNetV2 model...
  glioma: pred=glioma, conf=0.999, unc=0.0049  <-- MISCLASSIFIED, note this before comparing to EfficientNetB3
  meningioma: pred=meningioma, conf=0.971, unc=0.0339  <-- MISCLASSIFIED, note this before comparing to EfficientNetB3
  notumor: pred=notumor, conf=1.000, unc=0.0000  <-- MISCLASSIFIED, note this before comparing to EfficientNetB3
  pituitary: pred=pituitary, conf=0.995, unc=0.0057  <-- MISCLASSIFIED, note this before comparing to EfficientNetB3

All 4 images loaded. Starting XAI...



## 1. Grad-CAM++ (auto-detects the target conv layer -- verify the printed layer name)

In [ ]:
print("Grad-CAM++ (corrected: manual layer chaining, no nested-model splicing)")
backbone            = mobilenet_model.layers[1]
global_avg_pool     = mobilenet_model.layers[2]
dropout_layer       = mobilenet_model.layers[3]
classification_head = mobilenet_model.layers[4]

print("Target layer: 'out_relu' == backbone's own final output (verified earlier), "
      "so no separate grad_model needed -- chaining layers manually instead, "
      "the same proven pattern as the MC Dropout cell.")

def gradcam_pp(img_p, class_idx):
    inp = tf.cast(img_p, tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        co = backbone(inp, training=False)      # == out_relu output
        tape.watch(co)
        x    = global_avg_pool(co)
        x    = dropout_layer(x, training=False)
        pred = classification_head(x)
        loss = pred[:, class_idx]
        g1 = tape.gradient(loss, co)
        g2 = tape.gradient(g1, co)
        g3 = tape.gradient(g2, co)
    del tape

    if g1 is None or g2 is None or g3 is None:
        raise RuntimeError(
            f"Still None (g1: {g1 is None}, g2: {g2 is None}, g3: {g3 is None}). "
            f"Something more fundamental is wrong -- stop here, do not proceed."
        )

    co=co[0]; g1=g1[0]; g2=g2[0]; g3=g3[0]
    s   = tf.reduce_sum(co, axis=(0,1))
    den = 2*g2 + s[None,None,:]*g3
    den = tf.where(den==0, tf.ones_like(den), den)
    w   = tf.reduce_sum((g2/den)*tf.nn.relu(g1), axis=(0,1))
    h   = tf.nn.relu(tf.reduce_sum(w*co, axis=-1)).numpy()
    raw_max = h.max()   # <-- YAHAN ADD KIYA: normalize se PEHLE raw magnitude capture karo

    h   = cv2.resize(h, (224,224))
    if h.max()>0: h=(h-h.min())/(h.max()-h.min())
    return h, raw_max   # <-- YAHAN CHANGE KIYA: raw_max bhi return karo

import matplotlib
cmap_jet = matplotlib.colormaps['jet']

for cls_name in CLASS_NAMES:
    d  = data[cls_name]
    h, raw_max = gradcam_pp(d['img_p'], d['pred'])   # <-- ab do values aa rahi hain
    hc = (cmap_jet(h)[:,:,:3]*255).astype(np.uint8)
    ov = cv2.addWeighted(d['img'], 0.55, hc, 0.45, 0)
    data[cls_name]['gradcam'] = ov
    nonzero_pct = (h > 0.01).sum() / h.size * 100

    print(f"  {cls_name}: raw h.max() before normalization = {raw_max:.6f}")
    if raw_max < 0.01:
        print(f"    *** WARNING: raw signal near-zero ({raw_max:.6f}) -- "
              f"post-normalize heatmap will be VISUALLY MISLEADING, not a real hotspot ***")
    print(f"  {cls_name} done -- non-zero: {nonzero_pct:.1f}%")

Grad-CAM++ (corrected: manual layer chaining, no nested-model splicing)
Target layer: 'out_relu' == backbone's own final output (verified earlier), so no separate grad_model needed -- chaining layers manually instead, the same proven pattern as the MC Dropout cell.


  glioma: raw h.max() before normalization = 0.001687
    *** WARNING: raw signal near-zero (0.001687) -- post-normalize heatmap will be VISUALLY MISLEADING, not a real hotspot ***
  glioma done -- non-zero: 90.3%
  meningioma: raw h.max() before normalization = 0.000000
    *** WARNING: raw signal near-zero (0.000000) -- post-normalize heatmap will be VISUALLY MISLEADING, not a real hotspot ***
  meningioma done -- non-zero: 0.0%
  notumor: raw h.max() before normalization = 0.000005
    *** WARNING: raw signal near-zero (0.000005) -- post-normalize heatmap will be VISUALLY MISLEADING, not a real hotspot ***
  notumor done -- non-zero: 98.3%
  pituitary: raw h.max() before normalization = 0.001881
    *** WARNING: raw signal near-zero (0.001881) -- post-normalize heatmap will be VISUALLY MISLEADING, not a real hotspot ***
  pituitary done -- non-zero: 98.0%


In [ ]:
# Diagnostic -- run this once, just for meningioma, before deciding anything
d = data['meningioma']
inp = tf.cast(d['img_p'], tf.float32)
with tf.GradientTape(persistent=True) as tape:
    co = backbone(inp, training=False)
    tape.watch(co)
    x    = global_avg_pool(co)
    x    = dropout_layer(x, training=False)
    pred = classification_head(x)
    loss = pred[:, d['pred']]
    g1 = tape.gradient(loss, co)
    g2 = tape.gradient(g1, co)
    g3 = tape.gradient(g2, co)
del tape

co_=co[0]; g1_=g1[0]; g2_=g2[0]; g3_=g3[0]
s   = tf.reduce_sum(co_, axis=(0,1))
den = 2*g2_ + s[None,None,:]*g3_
den = tf.where(den==0, tf.ones_like(den), den)
w   = tf.reduce_sum((g2_/den)*tf.nn.relu(g1_), axis=(0,1))
pre_relu = tf.reduce_sum(w*co_, axis=-1)

print("w stats -- min:", float(tf.reduce_min(w)), "max:", float(tf.reduce_max(w)))
print("pre-ReLU heatmap -- min:", float(tf.reduce_min(pre_relu)), "max:", float(tf.reduce_max(pre_relu)))
print("Fraction of pre-ReLU values > 0:", float(tf.reduce_mean(tf.cast(pre_relu > 0, tf.float32))))

w stats -- min: -0.1185515895485878 max: 0.013276887126266956
pre-ReLU heatmap -- min: -0.045811623334884644 max: -6.663551175734028e-05
Fraction of pre-ReLU values > 0: 0.0


## 2. LIME (fully model-agnostic -- no changes needed beyond the preprocessing swap already made above)

In [ ]:
print("LIME")
explainer = lime_image.LimeImageExplainer(random_state=SEED)

def predict_fn(images):
    p = preprocess_fn(images.astype(np.float32))
    return mobilenet_model(p, training=False).numpy()

for cls_name in CLASS_NAMES:
    d   = data[cls_name]
    exp = explainer.explain_instance(
        d['img'].astype(np.uint8), predict_fn,
        top_labels=4, hide_color=0, num_samples=300, random_seed=SEED
    )
    temp, mask = exp.get_image_and_mask(
        d['pred'], positive_only=False, num_features=6, hide_rest=False
    )
    ov = mark_boundaries(temp.astype(np.uint8), mask.astype(np.int_),
                         color=(1,0,0), outline_color=(0,0,0))
    data[cls_name]['lime'] = (ov*255).astype(np.uint8)
    print(f"  {cls_name} done")


LIME


  0%|          | 0/300 [00:00<?, ?it/s]

  glioma done


  0%|          | 0/300 [00:00<?, ?it/s]

  meningioma done


  0%|          | 0/300 [00:00<?, ?it/s]

  notumor done


  0%|          | 0/300 [00:00<?, ?it/s]

  pituitary done


## 3. Integrated Gradients (fully model-agnostic)

In [ ]:
print("Integrated Gradients")

def integrated_grads(model, img_p, class_idx, n_steps=20):
    baseline = np.zeros_like(img_p)
    attrs    = np.zeros((224,224,3), dtype=np.float64)
    for alpha in np.linspace(0.0, 1.0, n_steps):
        interp = tf.constant(baseline + alpha*(img_p-baseline), dtype=tf.float32)
        with tf.GradientTape() as tape:
            tape.watch(interp)
            score = model(interp, training=False)[0, class_idx]
        attrs += tape.gradient(score, interp).numpy()[0]
    attrs  = (img_p[0]-baseline[0]) * attrs / n_steps
    attr2d = np.sum(np.abs(attrs), axis=-1)
    if attr2d.max()>0:
        attr2d = (attr2d-attr2d.min())/(attr2d.max()-attr2d.min())
    return attr2d

for cls_name in CLASS_NAMES:
    d    = data[cls_name]
    attr = integrated_grads(mobilenet_model, d['img_p'], d['pred'])
    data[cls_name]['ig'] = attr
    print(f"  {cls_name} done")


Integrated Gradients
  glioma done
  meningioma done
  notumor done
  pituitary done


## 4. Final 4-Panel Figure + Method-Agreement Notes

In [ ]:
print("Building 4-panel figure")

fig, axes = plt.subplots(4, 4, figsize=(18, 18))
row_titles = ['Original MRI', 'Grad-CAM++', 'LIME', 'Integrated Gradients']

for col, cls_name in enumerate(CLASS_NAMES):
    d = data[cls_name]
    axes[0,col].imshow(d['img'])
    axes[0,col].set_title(
        f'{cls_name.capitalize()}\n'
        f'Pred: {CLASS_NAMES[d["pred"]]} ({d["conf"]:.1%})\n'
        f'Unc: {d["unc"]:.4f}', fontsize=9)
    axes[0,col].axis('off')
    axes[1,col].imshow(d['gradcam']); axes[1,col].set_title('Grad-CAM++', fontsize=9); axes[1,col].axis('off')
    axes[2,col].imshow(d['lime']); axes[2,col].set_title('LIME', fontsize=9); axes[2,col].axis('off')
    axes[3,col].imshow(d['img'])
    axes[3,col].imshow(d['ig'], cmap='hot', alpha=0.5, vmin=0, vmax=1)
    axes[3,col].set_title('Integrated Gradients', fontsize=9); axes[3,col].axis('off')

for row, title in enumerate(row_titles):
    axes[row,0].set_ylabel(title, fontsize=12, fontweight='bold', labelpad=10)

plt.suptitle('XAI Panel -- MobileNetV2 Brain Tumor Classification', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/xai_4panel_mobilenet.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nSaved to: {FIG_DIR}/xai_4panel_mobilenet.png")

print("\n" + "="*60)
print("REPORTING CHECKLIST -- fill this in by eye before writing it up")
print("="*60)
print("""
For each class, note in your own words (this notebook can't judge
localization quality for you):
  1. Do Grad-CAM++, LIME, and IG agree on the same anatomical region?
  2. Is that region anatomically plausible for the tumor type?
  3. Does the pituitary class show the same localization weakness found
     with EfficientNetB3, or does MobileNetV2 behave differently here?
     (This is a real empirical question -- don't assume the answer.)
  4. Was any of the 4 sample predictions actually WRONG for MobileNetV2
     even though it was correct for EfficientNetB3, or vice versa? Check
     the 'pred' vs true class printed in Cell 0 above.
""")


In [ ]:
# Diagnostic -- glioma corner-hotspot check, no logic changes, read-only
d = data['glioma']
inp = tf.cast(d['img_p'], tf.float32)
with tf.GradientTape(persistent=True) as tape:
    co = backbone(inp, training=False)
    tape.watch(co)
    x    = global_avg_pool(co)
    x    = dropout_layer(x, training=False)
    pred = classification_head(x)
    loss = pred[:, d['pred']]
    g1 = tape.gradient(loss, co)
    g2 = tape.gradient(g1, co)
    g3 = tape.gradient(g2, co)
del tape

co_=co[0]; g1_=g1[0]; g2_=g2[0]; g3_=g3[0]
print("Raw backbone spatial output shape:", co_.shape)  # e.g. (7,7,1280)

s   = tf.reduce_sum(co_, axis=(0,1))
den = 2*g2_ + s[None,None,:]*g3_
den = tf.where(den==0, tf.ones_like(den), den)
w   = tf.reduce_sum((g2_/den)*tf.nn.relu(g1_), axis=(0,1))
h_raw = tf.nn.relu(tf.reduce_sum(w*co_, axis=-1)).numpy()

print("\nRaw (pre-resize) heatmap grid:")
np.set_printoptions(precision=3, suppress=True)
print(h_raw)

print("\nMax value location (row, col) in raw grid:", np.unravel_index(np.argmax(h_raw), h_raw.shape))
print("Grid shape:", h_raw.shape, "-- corners are index (0,0), (0,-1), (-1,0), (-1,-1)")

# Also check: is this a background/skull-edge region in the ORIGINAL image,
# not brain tissue? Print pixel brightness at the actual image corners.
img = d['img']
print("\nOriginal image mean brightness -- top-left corner (32x32):", img[:32,:32].mean())
print("Original image mean brightness -- bottom-left corner (32x32):", img[-32:,:32].mean())
print("Original image mean brightness -- bottom-right corner (32x32):", img[-32:,-32:].mean())
print("Original image mean brightness -- center (32x32):", img[96:128,96:128].mean())

Raw backbone spatial output shape: (7, 7, 1280)

Raw (pre-resize) heatmap grid:
[[0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.001 0.   ]
 [0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.002 0.    0.    0.    0.002 0.001]
 [0.    0.    0.    0.    0.    0.001 0.   ]]

Max value location (row, col) in raw grid: (np.int64(5), np.int64(5))
Grid shape: (7, 7) -- corners are index (0,0), (0,-1), (-1,0), (-1,-1)

Original image mean brightness -- top-left corner (32x32): 4.0
Original image mean brightness -- bottom-left corner (32x32): 4.25390625
Original image mean brightness -- bottom-right corner (32x32): 4.232421875
Original image mean brightness -- center (32x32): 114.0341796875
